# CO_robustness — Robustness Monte Carlo

Robustness Monte Carlo for conformal VaR recalibration (Appendix H, Tables H.14–H.16).

Three studies:
1. **Small-sample q̂_V volatility** — 500 replications, T ∈ {250, 500, 1000, 2000, 5000}, 5 DGPs
2. **Calibration-fraction sensitivity** — f_c ∈ {0.40, …, 0.80} for small T
3. **Rolling q̂_V stability** — 200 replications, GARCH(1,1) with embedded 3× variance crisis

All DGPs use GARCH(1,1) with ω=1e-5, α=0.10, β=0.85. Forecaster assumes Normal innovations.

**Outputs:** `tab_h14_small_sample.tex`, `tab_h15_fc_sensitivity.tex`, `tab_h16_regime_stability.tex`, CSVs

In [ ]:
"""
Robustness Monte Carlo for Conformal VaR Recalibration
(Appendix H, Tables H.14-H.16)
"""
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, ttest_ind
import warnings, time
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

ALPHA       = 0.01
FC_DEFAULT  = 0.70
N_REP       = 500
N_REP_ROLL  = 200
GARCH_OMEGA = 1e-5
GARCH_ALPHA = 0.10
GARCH_BETA  = 0.85

SCRIPT_DIR = Path('.').resolve()
ROOT_DIR   = SCRIPT_DIR.parent


def simulate_garch(T, omega=GARCH_OMEGA, alpha1=GARCH_ALPHA,
                   beta1=GARCH_BETA, innov='t5'):
    sigma2 = np.zeros(T)
    r = np.zeros(T)
    sigma2[0] = omega / (1 - alpha1 - beta1)

    if innov == 'normal':
        eps = np.random.normal(size=T)
    elif innov == 't5':
        eps = stats.t.rvs(df=5, size=T) / np.sqrt(5.0 / 3.0)
    elif innov == 't3':
        eps = stats.t.rvs(df=3, size=T) / np.sqrt(3.0)
    elif innov == 'skewt3':
        t_draws = stats.t.rvs(df=3, size=T)
        u = np.random.uniform(size=T)
        eps = np.where(u < 0.75, -np.abs(t_draws), np.abs(t_draws))
        eps = (eps - eps.mean()) / eps.std()
    elif innov == 'mixnormal':
        u = np.random.uniform(size=T)
        eps = np.where(u < 0.95,
                       np.random.normal(0, 1, T),
                       np.random.normal(0, 5, T))
        eps = (eps - eps.mean()) / eps.std()
    else:
        raise ValueError(f"Unknown innovation type: {innov}")

    for t in range(1, T):
        sigma2[t] = omega + alpha1 * r[t-1]**2 + beta1 * sigma2[t-1]
        r[t] = np.sqrt(sigma2[t]) * eps[t]

    return r, sigma2


def simulate_garch_regime(T, crisis_start, crisis_end,
                          omega=GARCH_OMEGA, alpha1=GARCH_ALPHA,
                          beta1=GARCH_BETA, crisis_mult=3.0,
                          innov='t5'):
    sigma2 = np.zeros(T)
    r = np.zeros(T)
    sigma2[0] = omega / (1 - alpha1 - beta1)

    if innov == 't5':
        eps = stats.t.rvs(df=5, size=T) / np.sqrt(5.0 / 3.0)
    elif innov == 'normal':
        eps = np.random.normal(size=T)
    elif innov == 'skewt3':
        t_draws = stats.t.rvs(df=3, size=T)
        u = np.random.uniform(size=T)
        eps = np.where(u < 0.75, -np.abs(t_draws), np.abs(t_draws))
        eps = (eps - eps.mean()) / eps.std()
    else:
        eps = np.random.normal(size=T)

    for t in range(1, T):
        omega_t = omega * crisis_mult if crisis_start <= t < crisis_end else omega
        sigma2[t] = omega_t + alpha1 * r[t-1]**2 + beta1 * sigma2[t-1]
        r[t] = np.sqrt(sigma2[t]) * eps[t]

    return r, sigma2


def compute_normal_var(returns, window=250, alpha=ALPHA):
    T = len(returns)
    var_f = np.full(T, np.nan)
    for t in range(window, T):
        block = returns[t - window:t]
        mu = block.mean()
        sig = block.std(ddof=1)
        var_f[t] = -(mu + sig * norm.ppf(alpha))
    return var_f


def conformal_static(returns, var_forecasts, alpha=ALPHA, fc=FC_DEFAULT):
    valid = np.where(~np.isnan(var_forecasts))[0]
    if len(valid) < 20:
        return np.nan, np.nan, np.nan, np.nan

    n_cal = int(len(valid) * fc)
    cal_idx = valid[:n_cal]
    test_idx = valid[n_cal:]
    if len(test_idx) < 5:
        return np.nan, np.nan, np.nan, np.nan

    scores_cal = (-var_forecasts[cal_idx]) - returns[cal_idx]
    q_V = np.quantile(scores_cal, 1 - alpha)

    raw_pi = (returns[test_idx] < -var_forecasts[test_idx]).mean()

    corrected_var = q_V + var_forecasts[test_idx]
    corr_pi = (returns[test_idx] < -corrected_var).mean()
    green = (corr_pi * 250) <= 4

    return q_V, corr_pi, raw_pi, green


def rolling_conformal(returns, var_forecasts, alpha=ALPHA, window=250):
    valid = np.where(~np.isnan(var_forecasts))[0]
    if len(valid) < window + 10:
        return np.array([]), np.nan, np.array([])

    q_V_roll = []
    violations = []
    times = []

    for i in range(window, len(valid)):
        t = valid[i]
        cal_indices = valid[i - window:i]
        scores = (-var_forecasts[cal_indices]) - returns[cal_indices]
        q_V_t = np.quantile(scores, 1 - alpha)
        q_V_roll.append(q_V_t)
        times.append(t)
        corrected_var_t = q_V_t + var_forecasts[t]
        violations.append(returns[t] < -corrected_var_t)

    return (np.array(q_V_roll),
            np.mean(violations) if violations else np.nan,
            np.array(times))


# ═══════════════════════════════════════════════════════════════
# STUDY 1: Small-sample q̂_V volatility
# ═══════════════════════════════════════════════════════════════

def run_study1():
    print("=" * 72)
    print("STUDY 1: Small-sample q̂_V volatility")
    print(f"  {N_REP} replications, α = {ALPHA}, f_c = {FC_DEFAULT}")
    print("=" * 72)

    sample_sizes = [250, 500, 1000, 2000, 5000]
    dgps = {
        'Normal (correct)': 'normal',
        'Student-t(5)':     't5',
        'Student-t(3)':     't3',
        'Skewed-t(3)':      'skewt3',
        'Mix. Normals':     'mixnormal',
    }

    rows = []
    for dgp_name, innov in dgps.items():
        for T in sample_sizes:
            win = min(250, max(50, T // 5))
            t0 = time.time()
            q_Vs, corr_pis, raw_pis, greens = [], [], [], []

            for rep in range(N_REP):
                r, _ = simulate_garch(T, innov=innov)
                var_f = compute_normal_var(r, window=win)
                q_V, cp, rp, g = conformal_static(r, var_f, fc=FC_DEFAULT)
                if not np.isnan(q_V):
                    q_Vs.append(q_V)
                    corr_pis.append(cp)
                    raw_pis.append(rp)
                    greens.append(g)

            q_Vs = np.array(q_Vs)
            m, s = np.mean(q_Vs), np.std(q_Vs)
            cv = s / m if abs(m) > 1e-10 else np.nan
            elapsed = time.time() - t0

            row = dict(
                DGP=dgp_name, T=T, est_window=win,
                mean_qV=m, std_qV=s, CV=cv,
                mean_corr_pi=np.mean(corr_pis),
                mean_raw_pi=np.mean(raw_pis),
                green_pct=np.mean(greens) * 100,
                n_valid=len(q_Vs),
            )
            rows.append(row)

            print(f"  {dgp_name:20s}  T={T:5d}  "
                  f"q̂_V = {m:.4f} ± {s:.4f}  "
                  f"CV = {cv:.2f}  "
                  f"π̂_corr = {np.mean(corr_pis):.3f}  "
                  f"Green = {np.mean(greens)*100:4.0f}%  "
                  f"({elapsed:.1f}s)")

    df = pd.DataFrame(rows)
    df.to_csv(SCRIPT_DIR / 'study1_small_sample.csv', index=False)
    return df


# ═══════════════════════════════════════════════════════════════
# STUDY 2: f_c sensitivity for small T
# ═══════════════════════════════════════════════════════════════

def run_study2():
    print("\n" + "=" * 72)
    print("STUDY 2: f_c sensitivity for small samples")
    print(f"  {N_REP} replications, α = {ALPHA}")
    print("=" * 72)

    fc_values = [0.40, 0.50, 0.60, 0.70, 0.80]
    small_Ts = [250, 500, 1000]
    dgps = {
        'Student-t(5)': 't5',
        'Skewed-t(3)':  'skewt3',
    }

    rows = []
    for dgp_name, innov in dgps.items():
        for T in small_Ts:
            win = min(250, max(50, T // 5))
            for fc_val in fc_values:
                t0 = time.time()
                q_Vs, corr_pis, greens = [], [], []

                for rep in range(N_REP):
                    r, _ = simulate_garch(T, innov=innov)
                    var_f = compute_normal_var(r, window=win)
                    q_V, cp, rp, g = conformal_static(r, var_f, fc=fc_val)
                    if not np.isnan(q_V):
                        q_Vs.append(q_V)
                        corr_pis.append(cp)
                        greens.append(g)

                q_Vs = np.array(q_Vs)
                m, s = np.mean(q_Vs), np.std(q_Vs)
                elapsed = time.time() - t0

                row = dict(
                    DGP=dgp_name, T=T, fc=fc_val,
                    mean_qV=m, std_qV=s,
                    mean_corr_pi=np.mean(corr_pis),
                    green_pct=np.mean(greens) * 100,
                    n_valid=len(q_Vs),
                )
                rows.append(row)

                print(f"  {dgp_name:15s}  T={T:4d}  f_c={fc_val:.2f}  "
                      f"q̂_V = {m:.4f} ± {s:.4f}  "
                      f"π̂ = {np.mean(corr_pis):.3f}  "
                      f"Green = {np.mean(greens)*100:4.0f}%  "
                      f"({elapsed:.1f}s)")

    df = pd.DataFrame(rows)
    df.to_csv(SCRIPT_DIR / 'study2_fc_sensitivity.csv', index=False)
    return df


# ═══════════════════════════════════════════════════════════════
# STUDY 3: Rolling q̂_V stability across regime changes
# ═══════════════════════════════════════════════════════════════

def run_study3():
    print("\n" + "=" * 72)
    print("STUDY 3: Rolling q̂_V stability across regime changes")
    print(f"  {N_REP_ROLL} replications, T=5000, crisis at t=2000-2500")
    print("=" * 72)

    T_regime = 5000
    crisis_s, crisis_e = 2000, 2500
    roll_w = 250
    est_w = 250

    pre_end       = crisis_s
    post_start    = crisis_e + roll_w

    innov_configs = [
        ('Student-t(5)', 't5'),
        ('Skewed-t(3)',  'skewt3'),
    ]

    all_rows = []
    for innov_name, innov in innov_configs:
        pre_m, dur_m, post_m = [], [], []

        t0 = time.time()
        for rep in range(N_REP_ROLL):
            r, sig2 = simulate_garch_regime(
                T_regime, crisis_s, crisis_e,
                crisis_mult=3.0, innov=innov
            )
            var_f = compute_normal_var(r, window=est_w)
            qV_roll, _, times = rolling_conformal(r, var_f, window=roll_w)

            if len(qV_roll) == 0:
                continue

            mask_pre    = times < pre_end
            mask_during = (times >= crisis_s) & (times < crisis_e)
            mask_post   = times >= post_start

            if mask_pre.any():
                pre_m.append(np.mean(qV_roll[mask_pre]))
            if mask_during.any():
                dur_m.append(np.mean(qV_roll[mask_during]))
            if mask_post.any():
                post_m.append(np.mean(qV_roll[mask_post]))

        elapsed = time.time() - t0
        pre_m  = np.array(pre_m)
        dur_m  = np.array(dur_m)
        post_m = np.array(post_m)

        t_stat, p_val = ttest_ind(pre_m, post_m) if len(pre_m) > 1 and len(post_m) > 1 else (np.nan, np.nan)

        print(f"\n  DGP: {innov_name} ({elapsed:.1f}s)")
        print(f"    Pre-crisis   q̂_V mean: {np.mean(pre_m):.5f} ± {np.std(pre_m):.5f}")
        print(f"    During crisis q̂_V mean: {np.mean(dur_m):.5f} ± {np.std(dur_m):.5f}")
        print(f"    Post-crisis  q̂_V mean: {np.mean(post_m):.5f} ± {np.std(post_m):.5f}")
        print(f"    Spike ratio (during/pre):  {np.mean(dur_m)/np.mean(pre_m):.2f}x")
        print(f"    Pre vs Post t-test: t = {t_stat:.3f}, p = {p_val:.3f}")

        all_rows.append(dict(DGP=innov_name, period='Pre-crisis',
                             mean_qV=np.mean(pre_m), std_qV=np.std(pre_m)))
        all_rows.append(dict(DGP=innov_name, period='During crisis',
                             mean_qV=np.mean(dur_m), std_qV=np.std(dur_m)))
        all_rows.append(dict(DGP=innov_name, period='Post-crisis',
                             mean_qV=np.mean(post_m), std_qV=np.std(post_m),
                             t_stat=t_stat, p_val=p_val))

    df = pd.DataFrame(all_rows)
    df.to_csv(SCRIPT_DIR / 'study3_regime_stability.csv', index=False)
    return df


# ═══════════════════════════════════════════════════════════════
# LaTeX table generation (writes files)
# ═══════════════════════════════════════════════════════════════

def write_tab_h14(df1):
    dgp_labels = {
        'Normal (correct)': 'Normal (correct)',
        'Student-t(5)':     'Student-$t(5)$',
        'Student-t(3)':     'Student-$t(3)$',
        'Skewed-t(3)':      'Skewed-$t(3)$',
        'Mix. Normals':     'Mixture of Normals',
    }
    lines = []
    lines.append(r"""\begin{table}[H]
	\centering
	\caption{Small-sample behaviour of $\qV$ (500 replications, $\alpha=0.01$, $f_c=0.70$). The forecaster assumes Normal innovations regardless of the true DGP. Green\% denotes the percentage of replications satisfying the Basel Traffic Light Green zone (at most 4 violations over 250 observations, scaled to sample size).}
	\label{tab:small_sample_qV}
	\footnotesize
	\begin{tabular}{@{}llcccc@{}}
		\toprule
		DGP & $T$ & Mean $\qV$ & Std $\qV$ & $\hat\pi$ & Green\% \\
		\midrule""")

    prev_dgp = ""
    for _, row in df1.iterrows():
        dgp = row['DGP']
        if dgp != prev_dgp:
            if prev_dgp != "":
                lines.append(r"		\addlinespace")
            lines.append(f"		{dgp_labels.get(dgp, dgp)}")
            prev_dgp = dgp
        lines.append(
            f"		& {int(row['T']):,d}   "
            f"& {row['mean_qV']:.4f} & {row['std_qV']:.4f} "
            f"& {row['mean_corr_pi']:.3f} & {row['green_pct']:.0f} \\\\"
        )

    lines.append(r"""		\bottomrule
	\end{tabular}
	\begin{minipage}{\linewidth}\footnotesize
		\smallskip
		For the GARCH DGP, parameters are $\omega = 10^{-5}$,
		$\alpha_1 = 0.10$, $\beta_1 = 0.85$, with estimation
		window $\min(250, T/5)$. Other DGPs are i.i.d.
		The standard deviation of $\qV$ decreases monotonically with $T$
		across all DGPs (a 3--5$\times$ reduction from $T=250$ to $T=5{,}000$),
		confirming that estimation error in the conformal threshold is
		dominated by tail sparsity at small sample sizes and vanishes at
		the $O((\alpha T)^{-1/2})$ rate. At small sample sizes
		($T \leq 500$), corrected violation rates exceed the nominal level
		due to estimation noise in the empirical quantile, consistent with
		finite-sample variability in extreme tails.
	\end{minipage}
\end{table}""")

    tex = "\n".join(lines)
    (SCRIPT_DIR / 'tab_h14_small_sample.tex').write_text(tex)
    (ROOT_DIR / 'tab_h14_small_sample.tex').write_text(tex)
    print(f"  Wrote tab_h14_small_sample.tex")


def write_tab_h15(df2):
    lines = []
    lines.append(r"""\begin{table}[H]
	\centering
	\caption{Calibration fraction sensitivity in small samples (500 replications, $\alpha = 0.01$). The forecaster assumes Normal innovations under misspecified heavy-tailed DGPs. Green\% denotes the percentage of replications satisfying the Basel Traffic Light Green zone (scaled to sample size).}
	\label{tab:fc_small_sample}
	\footnotesize
	\begin{tabular}{@{}llcccc@{}}
		\toprule
		DGP & $(T, f_c)$ & Mean $\qV$ & Std $\qV$ & $\hat\pi$ & Green\% \\
		\midrule""")

    prev_dgp = ""
    prev_T = -1
    for _, row in df2.iterrows():
        dgp = row['DGP']
        T = int(row['T'])
        if dgp != prev_dgp:
            if prev_dgp != "":
                lines.append(r"		\addlinespace")
            tex_dgp = f"Student-$t(5)$" if 't(5)' in dgp else f"Skewed-$t(3)$"
            lines.append(f"		\\multicolumn{{6}}{{@{{}}l}}{{\\textit{{{tex_dgp}}}}} \\\\[2pt]")
            prev_dgp = dgp
            prev_T = -1
        if T != prev_T and prev_T != -1:
            lines.append(r"		\addlinespace")
        prev_T = T
        lines.append(
            f"		& ({T}, {row['fc']:.2f}) "
            f"& {row['mean_qV']:.4f} & {row['std_qV']:.4f} "
            f"& {row['mean_corr_pi']:.3f} & {row['green_pct']:.0f} \\\\"
        )

    lines.append(r"""		\bottomrule
	\end{tabular}

	\begin{minipage}{\linewidth}\footnotesize
		\smallskip
		Calibration fraction controls a bias--variance trade-off in tail estimation: smaller $f_c$ reduces estimation variance but introduces bias due to insufficient tail coverage, while larger $f_c$ improves quantile estimation at the cost of reduced effective test sample size. For $T \leq 500$, no calibration fraction yields stable Basel Green performance under misspecification, reflecting the $O((\alpha T)^{-1/2})$ variance scaling implied by tail sparsity. For $T \geq 1{,}000$, performance stabilises for $f_c \in [0.60, 0.80]$, supporting the practical recommendation $f_c \geq 0.60$.
	\end{minipage}
\end{table}""")

    tex = "\n".join(lines)
    (SCRIPT_DIR / 'tab_h15_fc_sensitivity.tex').write_text(tex)
    (ROOT_DIR / 'tab_h15_fc_sensitivity.tex').write_text(tex)
    print(f"  Wrote tab_h15_fc_sensitivity.tex")


def write_tab_h16(df3):
    lines = []
    lines.append(r"""\begin{table}[H]
	\centering
	\caption{Rolling $\qV$ stability across regime changes (200 replications, GARCH(1,1), $T = 5{,}000$, stress regime at $t = 2{,}000$--$2{,}500$ with $3\times$ variance multiplier, rolling window $w = 250$). Reported values are time averages of $\hat q_{V,t}$ within each period, averaged across replications. The $t$-test compares pre-stress ($t < 2{,}000$) and post-stress ($t \geq 2{,}750$) means.}
	\label{tab:regime_stability}
	\footnotesize
	\begin{tabular}{@{}llccc@{}}
		\toprule
		DGP & Period & Mean $\qV$ & Std & $t$-test (Pre vs Post) \\
		\midrule""")

    dgp_groups = {}
    for _, row in df3.iterrows():
        dgp = row['DGP']
        if dgp not in dgp_groups:
            dgp_groups[dgp] = []
        dgp_groups[dgp].append(row)

    first = True
    for dgp, rows_list in dgp_groups.items():
        if not first:
            lines.append(r"		\addlinespace")
        first = False
        tex_dgp = f"Student-$t(5)$" if 't(5)' in dgp else f"Skewed-$t(3)$"
        lines.append(f"		\\multicolumn{{5}}{{@{{}}l}}{{\\textit{{{tex_dgp}}}}} \\\\[2pt]")
        for r in rows_list:
            period = r['period'].replace('crisis', 'stress')
            t_col = ""
            if 'p_val' in r and not np.isnan(r.get('p_val', np.nan)):
                t_col = f"$t = {r['t_stat']:.2f}$, $p = {r['p_val']:.3f}$"
            lines.append(
                f"		& {period:15s} & {r['mean_qV']:.5f} & {r['std_qV']:.5f} "
                f"& {t_col} \\\\"
            )

    lines.append(r"""		\bottomrule
	\end{tabular}

	\begin{minipage}{\linewidth}\footnotesize
		\smallskip
		The rolling conformal threshold $\hat q_{V,t}$ increases by a factor of 2--3 during the stress regime and reverts to pre-stress levels once the stressed observations exit the rolling window. The absence of statistically significant differences between pre- and post-stress means ($p > 0.05$) confirms that the correction adapts to transient volatility shocks without inducing persistent drift.
	\end{minipage}
\end{table}""")

    tex = "\n".join(lines)
    (SCRIPT_DIR / 'tab_h16_regime_stability.tex').write_text(tex)
    (ROOT_DIR / 'tab_h16_regime_stability.tex').write_text(tex)
    print(f"  Wrote tab_h16_regime_stability.tex")


# ── Main ───────────────────────────────────────────────────────

t_total = time.time()

df1 = run_study1()
df2 = run_study2()
df3 = run_study3()

print("\n\nWriting LaTeX tables...")
write_tab_h14(df1)
write_tab_h15(df2)
write_tab_h16(df3)

print(f"\n{'='*72}")
print(f"Total runtime: {time.time() - t_total:.0f} seconds")
print(f"Outputs saved to: {SCRIPT_DIR}/")
print(f"Tables copied to: {ROOT_DIR}/")
print(f"{'='*72}")

## Robustness Summary (Table D.15)

Weighted CP (3 λ values), calibration fraction sensitivity (4 f_c values),
and rolling 250-day conformal correction. Chronos-Small across 24 assets, α = 0.01.

**Output:** `tab_robustness_summary.tex`, `tab_robustness_summary.csv`

In [ ]:
%run run_robustness_summary.py